# S24 · The inventory case study (the deliverable)

This is the Module 4 deliverable. A small online shop must choose an `(s, S)`
restock policy for one product. We estimate demand from past sales, simulate
several candidate policies, pick the cheapest one that keeps at least 95% of
customers served, and write a clear recommendation with its assumptions. It reads
like a short consulting report, not just code.

**New here? Read this once.**

- New to Python? Press play on each cell, top to bottom, and read the note above
  each one. The output tells the whole story.
- This notebook reuses the `(s, S)` simulation from notebook 2. If you skipped it,
  a quick look there will help.
- Already confident? Look for the cell marked **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy and matplotlib.
# Google Colab already ships both, so there is nothing to install.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                 # random numbers and arrays
import matplotlib.pyplot as plt    # drawing charts

# A seed makes the whole study reproducible.
np.random.seed(0)

print("Libraries ready.")

## The brief

An online shop sells one product. Stock costs money to hold, and running out loses
sales and annoys customers. Demand each day is random. **Management wants a restock
policy that serves at least 95% of demand at the lowest cost.**

Our plan: (1) estimate demand from past sales, (2) simulate a range of `(s, S)`
policies, (3) choose the cheapest one that clears 95%, and (4) recommend it, stating
our assumptions.

## Step 1 — estimate demand from past sales

In a real project you would load a sales-history file. Here we create one year of
daily sales (365 numbers) to stand in for it, then estimate the average demand from
that history. Using the data, not a guess, is what makes a recommendation credible
to a business.

In [ ]:
# Pretend this is one year of recorded daily sales (our "history").
sales_history = np.random.poisson(12, size=365)

# Estimate the average daily demand straight from the history.
average_daily_demand = sales_history.mean()

print("Days of history                 :", len(sales_history))
print("Estimated average demand per day:", round(average_daily_demand, 2), "units")
print("Busiest day in history          :", sales_history.max(), "units")

## Step 2 — look at the demand history

Always look at your data before modelling it. A histogram shows how demand is
spread: most days are near the average, a few are much busier. Those busy days are
exactly when a thin shelf runs out.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(sales_history, bins=range(0, sales_history.max() + 2), color="#2E75B6")
plt.axvline(average_daily_demand, color="#C0392B", linestyle="--",
            label="average demand")
plt.xlabel("units demanded in a day")
plt.ylabel("number of days")
plt.title("One year of daily demand")
plt.legend()
plt.show()

## Step 3 — the simulation function

This is the same `(s, S)` simulation as notebook 2, repeated here so this notebook
stands on its own. It plays the policy out for a number of days and reports the
average cost per day and the service level. You do not need to re-read every line;
the point is that it returns two numbers we can compare across policies.

In [ ]:
# Costs and settings for the case study.
holding_cost_per_unit_per_day = 1.0
stockout_penalty_per_unit = 5.0
number_of_days = 300
lead_time = 2

def simulate_inventory(reorder_point, order_up_to_level):
    """Play the (s, S) policy and return average cost per day and service level."""

    stock = order_up_to_level
    incoming_order_day = None
    incoming_order_amount = 0

    total_holding_cost = 0.0
    total_stockout_cost = 0.0
    total_demand = 0
    total_units_short = 0

    for day in range(number_of_days):

        # Receive an arriving order, if any.
        if incoming_order_day == day:
            stock = stock + incoming_order_amount
            incoming_order_day = None
            incoming_order_amount = 0

        # Random demand, drawn around the average we estimated from history.
        demand_today = np.random.poisson(average_daily_demand)
        total_demand = total_demand + demand_today

        # Sell what we can; record any shortfall.
        if demand_today <= stock:
            stock = stock - demand_today
        else:
            units_short = demand_today - stock
            total_units_short = total_units_short + units_short
            total_stockout_cost = total_stockout_cost + units_short * stockout_penalty_per_unit
            stock = 0

        # Holding cost on leftover stock.
        total_holding_cost = total_holding_cost + stock * holding_cost_per_unit_per_day

        # Reorder if at or below the reorder point and nothing is on the way.
        if stock <= reorder_point and incoming_order_day is None:
            incoming_order_amount = order_up_to_level - stock
            incoming_order_day = day + lead_time

    total_cost = total_holding_cost + total_stockout_cost
    service_level = (total_demand - total_units_short) / total_demand

    average_cost_per_day = total_cost / number_of_days
    return average_cost_per_day, service_level

print("The simulate_inventory function is defined.")

## Step 4 — try several candidate policies

We pick a handful of sensible `(s, S)` settings and simulate each one. We store the
cost and service level for every candidate so we can compare them. A plain for-loop
handles them one at a time.

In [ ]:
# A list of candidate (s, S) settings to try.
candidate_policies = [
    (10, 40),
    (15, 50),
    (20, 60),
    (25, 70),
    (30, 80),
    (35, 90),
]

# We will collect one row of results per candidate.
candidate_s = []
candidate_S = []
candidate_cost = []
candidate_service = []

for reorder_point, order_up_to_level in candidate_policies:
    cost, service = simulate_inventory(reorder_point, order_up_to_level)
    candidate_s.append(reorder_point)
    candidate_S.append(order_up_to_level)
    candidate_cost.append(cost)
    candidate_service.append(service)
    print("s =", reorder_point, " S =", order_up_to_level,
          " -> cost/day =", round(cost, 2),
          " service =", round(service * 100, 1), "%")

## Step 5 — find the cheapest policy that meets the 95% target

We go through the candidates and keep the one with the lowest cost **among those
that reach at least 95% service**. This is the operations-research mindset: not the
cheapest overall, and not the most reliable overall, but the cheapest that still
clears the bar management set.

In [ ]:
service_target = 0.95

best_index = None
best_cost = None

for i in range(len(candidate_policies)):
    meets_target = candidate_service[i] >= service_target
    if meets_target:
        # Keep this one if it is the first to qualify, or cheaper than the best so far.
        if best_cost is None or candidate_cost[i] < best_cost:
            best_cost = candidate_cost[i]
            best_index = i

if best_index is None:
    print("No candidate reached the 95% target - we would need to hold more stock.")
else:
    print("Recommended policy:")
    print("  reorder point s   =", candidate_s[best_index])
    print("  order-up-to    S  =", candidate_S[best_index])
    print("  cost per day      =", round(candidate_cost[best_index], 2))
    print("  service level     =", round(candidate_service[best_index] * 100, 1), "%")

## Step 6 — picture the trade-off

We plot service level against cost for every candidate. The green line is the 95%
target; the star is the policy we chose, the cheapest point sitting on or above the
line. Seeing all candidates at once makes the choice obvious and easy to defend.

In [ ]:
plt.figure(figsize=(8, 5))

# Each candidate as a point: cost on the x-axis, service on the y-axis.
plt.scatter(candidate_cost, np.array(candidate_service) * 100,
            color="#2E75B6", s=80, label="candidate policies")

# Mark the chosen policy with a star.
if best_index is not None:
    plt.scatter(candidate_cost[best_index], candidate_service[best_index] * 100,
                color="#C0392B", s=250, marker="*", zorder=5,
                label="chosen policy")

# The 95% service target line.
plt.axhline(95, color="#27AE60", linestyle="--", label="95% target")

plt.xlabel("average cost per day")
plt.ylabel("service level (%)")
plt.title("Cost vs service level for each candidate (s, S) policy")
plt.legend()
plt.show()

## Step 7 — the recommendation

We finish with a short, plain-English recommendation and, just as importantly, the
assumptions behind it. Stating your assumptions is what separates an applied
mathematician from someone who just ran some code: it tells the reader exactly when
to trust the answer and when to redo the study.

In [ ]:
print("RECOMMENDATION")
print("==============")
if best_index is not None:
    print("Use the (s, S) policy: reorder at s =", candidate_s[best_index],
          ", order up to S =", candidate_S[best_index], ".")
    print("Expected cost   : about", round(candidate_cost[best_index], 2), "per day.")
    print("Expected service:", round(candidate_service[best_index] * 100, 1),
          "% of demand met (target was 95%).")
else:
    print("None of the tried policies met 95% service; try larger s and S values.")

print()
print("ASSUMPTIONS WE MADE")
print("-------------------")
print("1. Daily demand is random and steady (a Poisson with the historical average).")
print("2. The demand pattern does not drift over time (no festival spikes or trends).")
print("3. Orders always arrive after a fixed lead time of", lead_time, "days.")
print("4. We judged each policy on the long run, not a single lucky stretch.")

### Stretch (optional) — search a finer grid

Our six candidates were hand-picked. If you already code, here is how you would do
it properly: sweep many `(s, S)` combinations, simulate each, and let the code find
the cheapest one clearing 95%. Same logic as before, just a wider net. (For a fair
head-to-head, each policy is judged on the same seeded demand.)

In [ ]:
service_target = 0.95

best_fine_cost = None
best_fine_s = None
best_fine_S = None

# Try every reorder point from 10 to 40, and every gap up to S from 20 to 50.
for s_value in range(10, 41, 5):
    for gap in range(20, 51, 10):
        S_value = s_value + gap
        # Re-seed so every candidate faces the same demand stream (a fair test).
        np.random.seed(0)
        cost, service = simulate_inventory(s_value, S_value)
        if service >= service_target:
            if best_fine_cost is None or cost < best_fine_cost:
                best_fine_cost = cost
                best_fine_s = s_value
                best_fine_S = S_value

print("Finer search result:")
print("  best s        =", best_fine_s)
print("  best S        =", best_fine_S)
print("  cost per day  =", round(best_fine_cost, 2))

## What you just did

You ran a complete small operations-research study: you estimated demand from data,
simulated several `(s, S)` policies, chose the cheapest one meeting a 95% service
target, drew the cost-versus-service trade-off, and wrote a recommendation with its
assumptions. That is the Module 4 deliverable: a model, an analysis, a chart, and a
clear, honest conclusion. It ties the whole module together, because tuning this
policy is the same "find the long-run behaviour of a system that hops between
states" idea you started with in notebook 1.